In [0]:
%skip
%run "/Workspace/Users/jogesh.rajiyan@axahealth.co.uk/Telephony Data Analysis"

In [0]:
%skip
%run "/Workspace/Users/jogesh.rajiyan@axahealth.co.uk/MOL Data Analysis"

In [0]:
%sql
DROP TABLE IF EXISTS axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions;
CREATE TABLE axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions AS
SELECT ContactChannel
, conversationid
, PolicyNumber
, CustomerMessageTimestamp
, CustomerMessageSequenceNumberInCase AS TelephonyCustomerMessageSequence
, NULL AS MOLCustomerMessageSequence
, customermessage
, ResponseTimestamp
, AgentMessageTimestamp
, AgentMessageSequenceNumberInCase AS TelephonyAgentMessageSequence
, NULL AS MOLAgentMessageSequence
, agentmessage
, ResponseAgentId
, AgentName
, HasTransferFlag
, HandleTimeSecs
, PreviousMessageTimestamp AS Telephony_PreviousMessageTime
, NULL AS MOL_PreviousMessageTime
, GapFromPreviousMessageSecs AS Telephony_PreviousMessageGapSecs
, NULL AS MOL_PreviousMessageGapSecs
, ClaimNumber
, MembershipNumber
, ConversationStartTimestamp
, MemberId
, StandardisedSegment
, PolicySubType
, CurrentCondition
, CurrentConditionCategory
, System
, MSKClaim
, DaysSinceClaimOpened
, DaysSincePreviousContact AS Telephony_DaysSincePreviousContact
, NULL AS MOL_DaysSincePreviousContact
, ClaimantAge_ClaimOpenDate
, AgeCurrent
, JoinDate
, Tenure_Years
, Relationship
, PremiumAnnualGrossIPT
, PremiumAnnualNetIPT
, Gender
, CancellationDate
, UkRegion
, ClaimTotalPaid
, ExGratiaAmountPaid
, ComplaintArea
, ComplaintReceiptDate
, VulnerableCustomer
FROM axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_telephony_interaction_analytics
UNION
SELECT ContactChannel
, casenumber as conversationid
, PolicyNumber
, CustomerMessageTimestamp
, NULL AS TelephonyCustomerMessageSequence
, CustomerMessageSequenceNumberInCase AS MOLCustomerMessageSequence
, customermessage
, ResponseTimestamp
, AgentMessageTimestamp
, NULL AS TelephonyAgentMessageSequence
, AgentMessageSequenceNumberInCase AS MOLAgentMessageSequence
, agentmessage
, ResponseAgentId
, AgentName
, HasTransferFlag
, HandleTimeSecs
, NULL AS Telephony_PreviousMessageTime
, PreviousMessageTimestamp AS MOL_PreviousMessageTime
, NULL AS Telephony_PreviousMessageGapSecs
, GapFromPreviousMessageSecs AS MOL_PreviousMessageGapSecs
, ClaimNumber
, MembershipNumber
, ConversationStartTimestamp
, MemberId
, StandardisedSegment
, PolicySubType
, CurrentCondition
, CurrentConditionCategory
, System
, MSKClaim
, DaysSinceClaimOpened
, NULL AS Telephony_DaysSincePreviousContact
, DaysSincePreviousContact AS MOL_DaysSincePreviousContact
, ClaimantAge_ClaimOpenDate
, AgeCurrent
, JoinDate
, Tenure_Years
, Relationship
, PremiumAnnualGrossIPT
, PremiumAnnualNetIPT
, Gender
, CancellationDate
, UkRegion
, ClaimTotalPaid
, ExGratiaAmountPaid
, ComplaintArea
, ComplaintReceiptDate
, VulnerableCustomer
FROM axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_mol_interaction_analytics


In [0]:
%sql
SELECT claimnumber, COUNT(DISTINCT ContactChannel) FROM axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions GROUP BY 1 HAVING COUNT(DISTINCT ContactChannel) > 1

In [0]:
%sql
SELECT * FROM axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions WHERE ClaimNumber = 'AXACLM1801228'

In [0]:
%sql
Create or replace temporary TABLE ClaimCountRank as
SELECT
  ContactChannel,
  ConversationId,
  claimnumber,
  COUNT(claimnumber) AS claim_count,
  RANK() OVER (
      PARTITION BY ContactChannel, ConversationId
      ORDER BY COUNT(claimnumber) DESC, claimnumber ASC
    ) AS rnk
FROM
  axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions
GROUP BY
  ContactChannel,
  ConversationId,
  claimnumber;

  SELECT * FROM ClaimCountRank;

In [0]:
%sql
Create or replace temporary TABLE ClaimCountRank1 as
select
  *
from
  ClaimCountRank
where
  rnk = 1;

select
  *
from
  ClaimCountRank1

In [0]:
%sql
Create or replace temporary table channelsclaims_dedup as
select distinct
  a.*
from
  axahealth_dataplatform_pd_lab.jogesh_rajiyan_axahealth.fcr_cross_channel_interactions a
    inner join ClaimCountRank1 b
      on a.conversationid = b.conversationid
      and a.claimnumber = b.claimnumber;

select
  *
from
  channelsclaims_dedup

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE combinedchannelsContactSequence AS
SELECT *
, row_number() OVER (PARTITION BY claimnumber ORDER BY customermessageTimestamp) AS MessageSequence
, LAG(CustomerMessageTimestamp) OVER(PARTITION BY claimnumber order by customermessagetimestamp) AS PreviousMessageTimestamp
, timestampdiff(second,  LAG(CustomerMessageTimestamp) OVER(PARTITION BY claimnumber order by customermessagetimestamp), CustomerMessageTimestamp) AS GapFromPreviousMessageSecs
, timestampdiff(second,  LAG(CustomerMessageTimestamp) OVER(PARTITION BY claimnumber order by customermessagetimestamp), CustomerMessageTimestamp)/3600.0 AS GapHours
FROM channelsclaims_dedup

In [0]:
%sql
SELECT * FROM combinedchannelsContactSequence WHERE claimnumber = 'AXACLM1801228'

# Repeat Contact Sequencing

Contact -> Episode -> Sequence -> Customer Journey

**Contact** - Was this interaction Handled?

**Episode** - Did the customer continue contacting within immediate contact window?

**Sequence** - Did the customer return shortly after the episode ended?

**Longer-term repeat** - Did the customer come back later?


Contact	Time	    Gap	Episode	Sequence

C1	    Mon 09:00	—	E1	    S1

C2	    Mon 15:00	6h	E1	    S1

C3	    Tue 10:00	19h	E1	    S1

C4	    Wed 09:00	23h	E1	    S1

C5	    Thu 10:00	25h	E2	    S1

C6	    Fri 09:00	23h	E2	    S1

C7	    Mon 10:00	73h	E3	    S2

With a 24Hr Episode Threshold:

C1 - C4 -> One Episode 

C5 -C6 -> another episode

C7 -> another episode

With 48Hr Threshold:

E1 + E2  - Sequence S1

E3 - Sequence S2

**Pattern A - Immediate Repeat**

Contact -> Contact -> Contact = 1 Episode

This is useful for identifying failure to resolve within the same interaction period.

**Pattern B - Episode -to- episode repeat**

Episode 1 -> 25h -> Episode 2

First episode ended according to your episode definition, but the customer returned very soon afterwards.

Can be evidence for:
- unresolved demand
- delayed resolution
- follow-up requirement
- customer having to contact again
- operation handoffs
- claims/processs dependencies

**Pattern C - Genuine later re-contact**

Episode 1 -> 2 weeks -> Episode 2

This may be a completely different customer need.


## Distribution Analysis to identify Gaps between Contacts for Episode Classification

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW contact_gap_distribution AS 
SELECT conversationid
,Claimnumber
,previousmessagetimestamp
,CustomerMessageTimestamp
,MessageSequence
,GapFromPreviousMessageSecs
,GapFromPreviousMessageSecs/3600.0 AS GapHours
FROM combinedchannelsContactSequence
WHERE GapFromPreviousMessageSecs IS NOT NULL AND GapFromPreviousMessageSecs >=0;

In [0]:
%sql
SELECT COUNT(*) AS GapCount
    ,ROUND(MIN(GapHours),2) AS MinHours
    ,ROUND(percentile_approx(GapHours,0.01,1000),2) AS P01
    ,ROUND(percentile_approx(GapHours,0.05,1000),2) AS P05
    ,ROUND(percentile_approx(GapHours,0.10,1000),2) AS P10
    ,ROUND(percentile_approx(GapHours,0.25,1000),2) AS P25
    ,ROUND(percentile_approx(GapHours,0.50,1000),2) AS P50
    ,ROUND(percentile_approx(GapHours,0.75,1000),2) AS P75
    ,ROUND(percentile_approx(GapHours,0.90,1000),2) AS P90
    ,ROUND(percentile_approx(GapHours,0.95,1000),2) AS P95
    ,ROUND(percentile_approx(GapHours,0.99,1000),2) AS P99
    ,ROUND(MAX(GapHours),2) AS MaxHours
FROM contact_gap_distribution;

In [0]:
from pyspark.sql import functions as F
contact_gaps = spark.table('contact_gap_distribution')
histogram = contact_gaps.withColumn("GapBucket",F.when(F.col("GapHours") < 1,0).otherwise(F.floor(F.col("GapHours")))).groupBy("GapBucket").count() \
    .withColumnRenamed("count","GapCount") \
    .orderBy("GapBucket")

display(histogram.filter(F.col("GapBucket")<=72))

In [0]:
import matplotlib.pyplot as plt

hist_pd = histogram.filter(F.col("GapBucket")<=72).toPandas()

plt.figure(figsize=(14,6))
plt.bar(
    hist_pd["GapBucket"],
    hist_pd["GapCount"],
    width=0.8
)
plt.axvline(
    24,
    linestyle = "--",
    linewidth=2,
    label="Current Episode = 24h"
)
plt.axvline(
    48,
    linestyle="--",
    linewidth=2,
    label="Current Sequence = 48h"
)

plt.xlabel("Gap from previous customer contact (hours)")
plt.ylabel("Number of Gaps")
plt.title("Cross Channel Contact Gap Distribution")

plt.legend()
plt.grid(axis="y",alpha=0.3)

plt.show()

In [0]:
thresholds = [2,4,6,8,12,16,18,20,24,30,36,48,72]

total_gaps = contact_gaps.count()

results = []

for threshold in thresholds:
    within = contact_gaps.filter(F.col("GapHours")<=threshold).count()
    results.append(
        (
            threshold,
            total_gaps,
            within,
            round(100*within/total_gaps,2)
        )
    )

threshold_df = spark.createDataFrame(results,
                                     [
                                         "ThresholdHours",
                                         "TotalGaps",
                                         "GapsWithinThreshold",
                                         "PercentageWithinThreshold"
                                     ]
                                     )

display(threshold_df.orderBy("ThresholdHours"))

In [0]:
from pyspark.sql.window import Window

thresholds = [6,12,18,20,24,30,36,48,72]

results = []

for threshold in thresholds:
    w = Window.partitionBy("ConversationId","ClaimNumber").orderBy("CustomerMessageTimestamp","MessageSequence")
    df = contact_gaps.withColumn("NewEpisode", F.when(F.col("PreviousMessageTimestamp").isNull() | (F.col("GapHours")>=threshold),1).otherwise(0)) \
                .withColumn("EpisodeId", F.sum("NewEpisode").over(w))
    episode_stats = df.groupBy("ConversationId","ClaimNumber","EpisodeId") \
                        .agg(F.count("*").alias("ContactsPerEpisode"))
    summary = episode_stats.agg(F.count("*").alias("EpisodeCount"), F.avg("ContactsPerEpisode").alias("AvgContactsPerEpisode"),F.expr("percentile_approx(ContactsPerEpisode,0.5)").alias("MedianContactsPerEpisode"),F.sum(F.when(F.col("ContactsPerEpisode") == 1,1).otherwise(0)).alias("SingleContactEpisodes"),F.count("*").alias("TotalEpisodes")).withColumn("SingleContactEpisodePct",F.round(F.col("SingleContactEpisodes")/F.col("TotalEpisodes")*100,2)).withColumn("EpisodeThresholdHours",F.lit(threshold))

    results.append(summary)

episode_sensitivity = results[0]

for r in results[1:]:
    episode_sensitivity = episode_sensitivity.unionByName(r)

display(episode_sensitivity.select("EpisodeThresholdHours","EpisodeCount","AvgContactsPerEpisode","MedianContactsPerEpisode","SingleContactEpisodePct").orderBy("EpisodeThresholdHours"))

In [0]:
threshold_results = []

for threshold in thresholds:
    w = Window.partitionBy("ConversationId","ClaimNumber").orderBy("CustomerMessageTimestamp","MessageSequence")
    df = contact_gaps.withColumn("NewEpisode", F.when(F.col("PreviousMessageTimestamp").isNull() | (F.col("GapHours")>=threshold),1).otherwise(0)) \
                .withColumn("EpisodeId", F.sum("NewEpisode").over(w))
    stats = df.agg(
        F.count("*").alias("TotalContacts"),
        F.sum("NewEpisode").alias("NewEpisodeStarts")
    ).withColumn(
        "EpisodeStartPct",
        F.round(
            F.col("NewEpisodeStarts") /
            F.col("TotalContacts") * 100,
            2
        )
    ).withColumn(
        "EpisodeThresholdHours",
        F.lit(threshold)
    )

    threshold_results.append(stats)

episode_start_sensitivity = threshold_results[0]

for r in threshold_results[1:]:
    episode_start_sensitivity = episode_start_sensitivity.unionByName(r)

display(
    episode_start_sensitivity.select(
        "EpisodeThresholdHours",
        "TotalContacts",
        "NewEpisodeStarts",
        "EpisodeStartPct"
    ).orderBy("EpisodeThresholdHours")
)

In [0]:
final_episode_sensitivity = episode_sensitivity.join(
    episode_start_sensitivity,
    on="EpisodeThresholdHours",
    how="inner"
).select(
    "EpisodeThresholdHours",
    "EpisodeCount",
    "AvgContactsPerEpisode",
    "MedianContactsPerEpisode",
    "SingleContactEpisodePct",
    "NewEpisodeStarts",
    "EpisodeStartPct"
    ).orderBy("EpisodeThresholdHours")

display(final_episode_sensitivity)

In [0]:
import matplotlib.pyplot as plt

pdf = final_episode_sensitivity.toPandas()

fig, ax = plt.subplots(figsize=(10,6))

ax.plot(
    pdf["EpisodeThresholdHours"],
    pdf["EpisodeCount"],
    marker="o",
    label="Episode Count"
)

ax.set_xlabel("Episode Threshold (hours)")
ax.set_ylabel("Number of Episodes")
ax.set_title("Cross Channel Episode Threshold Sensitivity")
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

In [0]:
import matplotlib.pyplot as plt

pdf = final_episode_sensitivity.toPandas()

fig, ax = plt.subplots(figsize=(10,6))

ax.plot(
    pdf["EpisodeThresholdHours"],
    pdf["SingleContactEpisodePct"],
    marker="o",
    label="Single-Contact Episode %"
)

ax.set_xlabel("Episode Threshold (hours)")
ax.set_ylabel("Percentage")
ax.set_title("Cross Channel Episode Fragmentation by Threshold")
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE ContactEpisodeTable AS
WITH ContactCandidate AS(
    SELECT DISTINCT *
,CASE WHEN GapFromPreviousMessageSecs IS NULL THEN 1
    WHEN GapFromPreviousMessageSecs >= 24 * 60 * 60 THEN 1
    ELSE 0
END AS NewContactCandidate
FROM combinedchannelsContactSequence
)
SELECT * 
,SUM(NewContactCandidate) OVER(PARTITION BY claimnumber ORDER BY CustomerMessageTimestamp, MessageSequence ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ContactEpisode
FROM ContactCandidate
ORDER BY claimnumber,CustomerMessageTimestamp
;
SELECT * FROM ContactEpisodeTable 
WHERE claimnumber = "AXACLM1801228" ;

## Distribution Analysis to identify Gaps between Contacts for Sequence Classification

In [0]:
contactepisodetable = spark.table("contactepisodetable")

episode_summary = (
    contactepisodetable
    .groupBy(
        "ConversationId",
        "ClaimNumber",
        "ContactEpisode"
    )
    .agg(
        F.min("CustomerMessageTimestamp").alias("EpisodeStart"),
        F.max("CustomerMessageTimestamp").alias("EpisodeEnd"),
        F.count("*").alias("ContactsInEpisode")
    )
)

display(
    episode_summary
    .orderBy("ConversationId", "ClaimNumber", "EpisodeStart")
)

In [0]:
episode_window = Window.partitionBy(
    "ConversationId",
    "ClaimNumber"
).orderBy(
    "EpisodeStart"
)

episode_gaps = (
    episode_summary
    .withColumn(
        "PreviousEpisodeEnd",
        F.lag("EpisodeEnd").over(episode_window)
    )
    .withColumn(
        "GapHoursFromPreviousEpisode",
        (
            F.col("EpisodeStart").cast("long")
            - F.col("PreviousEpisodeEnd").cast("long")
        ) / 3600
    )
)

display(
    episode_gaps
    .select(
        "ConversationId",
        "ClaimNumber",
        "ContactEpisode",
        "EpisodeStart",
        "EpisodeEnd",
        "PreviousEpisodeEnd",
        "GapHoursFromPreviousEpisode"
    )
    .orderBy("ConversationId", "ClaimNumber", "EpisodeStart")
)


In [0]:
sequence_gap_distribution = (
    episode_gaps
    .filter(
        F.col("GapHoursFromPreviousEpisode").isNotNull()
    )
    .withColumn(
        "GapBucketHours",
        F.floor("GapHoursFromPreviousEpisode")
    )
    .groupBy("GapBucketHours")
    .count()
    .withColumnRenamed("count", "GapCount")
    .orderBy("GapBucketHours")
)

display(sequence_gap_distribution)


In [0]:
import matplotlib.pyplot as plt

gap_pd = (
    sequence_gap_distribution
    .filter(F.col("GapBucketHours") <= 168)
    .toPandas()
)

plt.figure(figsize=(14, 6))

plt.bar(
    gap_pd["GapBucketHours"],
    gap_pd["GapCount"],
    width=0.8
)

plt.axvline(
    48,
    linestyle="--",
    label="Current Sequence = 48h"
)

plt.xlabel("Gap between Episodes (hours)")
plt.ylabel("Number of Episode Gaps")
plt.title("Cross Channel Inter-Episode Gap Distribution")

plt.legend()
plt.tight_layout()
plt.show()


In [0]:
sequence_thresholds = [
    24,
    36,
    48,
    60,
    72,
    96,
    120,
    168
]

sequence_results = []

for threshold in sequence_thresholds:

    result = (
        episode_gaps
        .filter(
            F.col("GapHoursFromPreviousEpisode").isNotNull()
        )
        .withColumn(
            "NewSequence",
            F.when(
                F.col("GapHoursFromPreviousEpisode") >= threshold,
                1
            ).otherwise(0)
        )
        .agg(
            F.count("*").alias("TotalEpisodeGaps"),
            F.sum("NewSequence").alias("NewSequenceStarts")
        )
        .withColumn(
            "SequenceStartPct",
            F.round(
                F.col("NewSequenceStarts") /
                F.col("TotalEpisodeGaps") * 100,
                2
            )
        )
        .withColumn(
            "SequenceThresholdHours",
            F.lit(threshold)
        )
    )

    sequence_results.append(result)

sequence_sensitivity = sequence_results[0]

for r in sequence_results[1:]:
    sequence_sensitivity = sequence_sensitivity.unionByName(r)

display(
    sequence_sensitivity
    .select(
        "SequenceThresholdHours",
        "TotalEpisodeGaps",
        "NewSequenceStarts",
        "SequenceStartPct"
    )
    .orderBy("SequenceThresholdHours")
)


In [0]:
sequence_sensitivity_results = []

for threshold in sequence_thresholds:

    w = Window.partitionBy(
        "ConversationId",
        "ClaimNumber"
    ).orderBy("EpisodeStart")

    df = (
        episode_gaps
        .withColumn(
            "NewSequence",
            F.when(
                F.col("PreviousEpisodeEnd").isNull(),
                1
            ).when(
                F.col("GapHoursFromPreviousEpisode") >= threshold,
                1
            ).otherwise(0)
        )
        .withColumn(
            "SequenceNumber",
            F.sum("NewSequence").over(w)
        )
    )

    sequence_summary = (
        df
        .groupBy(
            "ConversationId",
            "ClaimNumber",
            "SequenceNumber"
        )
        .agg(
            F.count("*").alias("EpisodesInSequence")
        )
    )

    result = (
        sequence_summary
        .agg(
            F.count("*").alias("SequenceCount"),

            F.round(
                F.avg("EpisodesInSequence"),
                3
            ).alias("AvgEpisodesPerSequence"),

            F.expr(
                "percentile_approx(EpisodesInSequence, 0.5)"
            ).alias("MedianEpisodesPerSequence"),

            F.sum(
                F.when(
                    F.col("EpisodesInSequence") == 1,
                    1
                ).otherwise(0)
            ).alias("SingleEpisodeSequences")
        )
        .withColumn(
            "SequenceThresholdHours",
            F.lit(threshold)
        )
        .withColumn(
            "SingleEpisodeSequencePct",
            F.round(
                F.col("SingleEpisodeSequences") /
                F.col("SequenceCount") * 100,
                2
            )
        )
    )

    sequence_sensitivity_results.append(result)

sequence_sensitivity_final = sequence_sensitivity_results[0]

for r in sequence_sensitivity_results[1:]:
    sequence_sensitivity_final = (
        sequence_sensitivity_final.unionByName(r)
    )

display(
    sequence_sensitivity_final
    .select(
        "SequenceThresholdHours",
        "SequenceCount",
        "AvgEpisodesPerSequence",
        "MedianEpisodesPerSequence",
        "SingleEpisodeSequencePct"
    )
    .orderBy("SequenceThresholdHours")
)


In [0]:
seq_pd = (
    sequence_sensitivity_final
    .orderBy("SequenceThresholdHours")
    .toPandas()
)

plt.figure(figsize=(12, 6))

plt.plot(
    seq_pd["SequenceThresholdHours"],
    seq_pd["AvgEpisodesPerSequence"],
    marker="o"
)

plt.axvline(
    48,
    linestyle="--",
    label="48h"
)

plt.xlabel("Sequence Threshold (hours)")
plt.ylabel("Average Episodes per Sequence")
plt.title("Cross Channel Sequence Threshold Sensitivity")

plt.legend()
plt.tight_layout()
plt.show()

## Episode Analysis

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLevel AS
SELECT CONCAT_WS(', ', transform(array_sort(collect_set(struct(CustomerMessageTimestamp,ConversationId))), x -> x.ConversationId)) AS Conversations
,MAX(policynumber) AS policynumber
,claimnumber
,memberid
,MAX(membershipnumber) AS membershipnumber
,ContactEpisode
,MAX(standardisedsegment) AS standardisedsegment
,MAX(policysubtype) AS policysubtype
,MAX(currentcondition) AS currentcondition
,MAX(currentconditioncategory) AS currentconditioncategory
,MAX(system) AS system
,MAX(relationship) AS relationship
,MAX(gender) AS gender
,MAX(ComplaintArea) AS ComplaintArea
,MAX(ComplaintReceiptDate) AS ComplaintReceiptDate
,MAX(VulnerableCustomer) AS VulnerableCustomer
,MAX(UkRegion) AS UkRegion
,MAX(CancellationDate) AS CancellationDate
,MAX(MSKClaim) AS MSKClaim
,MAX(ClaimantAge_ClaimOpenDate) AS ClaimantAge_ClaimOpenDate
,MAX(AgeCurrent) AS AgeCurrent
,MAX(JoinDate) AS JoinDate
,MAX(Tenure_Years) AS Tenure_Years
,ROUND(AVG(DaysSinceClaimOpened)) AS DaysSinceClaimOpened
,COUNT(CustomerMessage) AS MessageCount
,MAX(conversationstarttimestamp) AS conversationstarttimestamp
,MIN(CustomerMessageTimeStamp) AS EpisodeStart
,MAX(CustomerMessageTimestamp) AS EpisodeEnd
,CONCAT_WS('\n', transform(array_sort(collect_list(struct(CustomerMessageTimestamp, MessageSequence, CustomerMessage, ContactChannel))), x -> CONCAT('[',x.ContactChannel, '] ', x.CustomerMessage))) AS CustomerEpisodeConversation
,CONCAT_WS('\n', transform(array_sort(collect_set(struct(AgentMessageTimestamp,MessageSequence, AgentMessage,AgentName,ContactChannel))), x -> CONCAT('[',x.ContactChannel, '] ', CAST(x.AgentMessageTimestamp AS STRING),' ',COALESCE(x.AgentName, 'Unknown Agent'), ': ', COALESCE(x.AgentMessage,'')))) AS AgentEpisodeConversation
,CONCAT_WS(', ', transform(array_sort(collect_set(struct(AgentMessageTimestamp, MessageSequence, ResponseAgentId))), x -> x.ResponseAgentId)) AS AgentResponsible
,COUNT(DISTINCT AgentMessage) AS AgentResponseCount
,CASE WHEN SUM(HandleTimeSecs) > 0 THEN SUM(HandleTimeSecs) ELSE 0 END AS ConversationTimeInSecs
,AVG(PremiumAnnualGrossIPT) AS PremiumAnnualGrossIPT
,AVG(PremiumAnnualNetIPT) AS PremiumAnnualNetIPT
,AVG(ClaimTotalPaid) AS ClaimTotalPaid
,AVG(ExGratiaAmountPaid) AS ExGratiaAmountPaid
FROM ContactEpisodeTable
GROUP BY ALL
ORDER BY claimnumber, ContactEpisode
;
SELECT * FROM EpisodeLevel WHERE claimnumber = "AXACLM1801228";

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLinks AS
SELECT *
, LAG(ContactEpisode) OVER(PARTITION BY claimnumber ORDER BY ContactEpisode) AS PreviousContactEpisode
, LAG(EpisodeEnd) OVER(PARTITION BY claimnumber ORDER BY ContactEpisode) AS PreviousEpisodeEnd
FROM EpisodeLevel
;

SELECT * FROM EpisodeLinks WHERE claimnumber = 'AXACLM1801228';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeLinksWithGap AS
SELECT *
, ROUND((UNIX_TIMESTAMP(EpisodeStart) - UNIX_TIMESTAMP(PreviousEpisodeEnd)) / 3600.0,2) AS GapHours
FROM EpisodeLinks
;
SELECT * FROM EpisodeLinksWithGap WHERE claimnumber = 'AXACLM1801228';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW EpisodeAnalysis AS
SELECT *
, CASE WHEN PreviousContactEpisode IS NULL THEN 0 ELSE 1 END AS HasPreviousEpisode
, CASE WHEN PreviousContactEpisode IS NULL THEN 'FIRST_CONTACT' 
        WHEN GapHours <= 60 THEN 'REPEAT_CANDIDATE'
        ELSE 'LONG_GAP_CANDIDATE'
    END AS RepeatCandidateType
FROM EpisodeLinksWithGap;

SELECT * FROM episodeanalysis WHERE claimnumber = "AXACLM1801228" ORDER BY ContactEpisode;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW ContactSequences AS
WITH SequenceFlags AS (
    SELECT *
    , CASE WHEN PreviousContactEpisode IS NULL THEN 1
        WHEN GapHours > 60 THEN 1 ELSE 0
        END AS NewSequenceFlag
    FROM episodeanalysis
),
SequenceNumbered AS (
    SELECT *
    , SUM(NewSequenceFlag) OVER(PARTITION BY claimnumber ORDER BY ContactEpisode ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ContactSequence
    FROM sequenceflags
)

SELECT * FROM sequencenumbered;

SELECT * FROM ContactSequences WHERE WHERE claimnumber = "AXACLM1801228" 

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW RepeatContactSequences AS

SELECT Conversations,
memberid,
claimnumber,
        ContactSequence,
        MIN(ContactEpisode) AS SequenceStartEpisode,
        MAX(ContactEpisode) AS SequenceEndEpisode,
        MIN(EpisodeStart) AS SequenceStart,
        MAX(EpisodeEnd) AS SequenceEnd,
        COUNT(*) AS EpisodeCount,
        SUM(MessageCount) AS MessageCount,
        ROUND((UNIX_TIMESTAMP(MAX(EpisodeEnd)) - UNIX_TIMESTAMP(MIN(EpisodeStart)))/3600.0,2) AS SequenceDurationHours
FROM ContactSequences
GROUP BY Conversations, ContactSequence,memberid,claimnumber
;

SELECT * FROM repeatcontactsequences WHERE claimnumber='AXACLM1801228'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY TABLE FinalCombinedChannels AS
SELECT cs.conversations
,cs.policynumber
,cs.claimNumber
,cs.memberid
,cs.membershipnumber
,cs.standardisedsegment
,cs.policysubtype
,cs.currentcondition
,cs.currentconditioncategory
,cs.system
,cs.relationship
,cs.gender
,cs.complaintarea
,cs.complaintreceiptdate
,cs.vulnerablecustomer
,cs.ukregion
,cs.cancellationdate
,cs.mskclaim
,cs.ClaimantAge_ClaimOpendate
,cs.Agecurrent
,cs.joindate
,cs.tenure_years
,cs.dayssinceclaimopened
,cs.ContactEpisode
,cs.ContactSequence
,cs.conversationstarttimestamp
,cs.EpisodeStart
,cs.EpisodeEnd
,cs.PreviousContactEpisode
,cs.PreviousEpisodeEnd
,cs.GapHours
,cs.conversationtimeinsecs
,cs.premiumannualgrossipt
,cs.premiumannualnetipt
,cs.claimtotalpaid
,cs.exgratiaamountpaid
,cs.MessageCount
,cs.AgentResponseCount
,cs.CustomerEpisodeConversation
,cs.AgentEpisodeConversation
,cs.AgentResponsible
,cs.HasPreviousEpisode
,cs.RepeatCandidateType
,CASE WHEN cs.HasPreviousEpisode = 1 THEN TRUE ELSE FALSE END AS IsRepeatContact
,CASE WHEN cs.ContactEpisode > rcs.sequenceStartEpisode THEN TRUE ELSE FALSE END AS IsRepeatWithinSequence
,CASE WHEN cs.ContactEpisode = rcs.SequenceStartEpisode THEN TRUE ELSE FALSE END AS IsSequenceStart
,rcs.SequenceStartEpisode
,rcs.SequenceEndEpisode
,rcs.SequenceEnd
,rcs.EpisodeCount
,rcs.MessageCount AS SequenceMessageCount
,rcs.SequenceDurationHours
FROM ContactSequences cs
LEFT JOIN RepeatContactSequences rcs
ON cs.Claimnumber = rcs.claimnumber
AND cs.ContactSequence = rcs.ContactSequence

;

SELECT * FROM finalcombinedchannels WHERE claimnumber = 'AXACLM1801228'